# Chapter 1 · Notebook 1 of 3 — Introducing Token Factory

**AI Agent Course** · Nebius Token Factory × NVIDIA

Over the three notebooks of this chapter, we build **our first controlled agentic loop**: a model that can request an action, have our Python code execute it, and use the result. This notebook covers the first two steps of that journey: **choosing a model** and **calling it from Python**. Notebook 2 adds the tool loop and Notebook 3 checks whether the loop is fast enough and cheap enough to be usable.

**Prerequisites:** Python 3.10+, basic familiarity with REST APIs. No GPU, no prior LLM experience assumed.

**By the end of this notebook you can:** navigate the Token Factory catalog, read a model card and pick a model for a job, and make chat completion calls from Python — single-turn, styled with system prompts, and multi-turn.

## 1. What is Token Factory?

Open-sourced models are those in which all ingredients - including the weights, documentation, and often research - is publically available. Open source models are particularly powerful because they can be ran on your own servers.

In practice, that takes serious capacity and skills - GPU clusters, serving stacks, scaling, monitoring - and most teams have better things to do. **Managed inference** providers like Nebius Token Factory close that gap: they deploy and operate the open-source models for you. You can interact with these cheaper (and often equally as powerful!) models all in the playground or with simple API calls.

**In this course, we use Token Factory's public models which available to all users and billed per token which makes monitoring cost simple.**

## 2. Setup

1. Go to [tokenfactory.nebius.com](https://tokenfactory.nebius.com) and create an account.
2. Open **Get API Key → Create API key** and copy it (you can't view it again later).

<details>
<summary>Show screenshot: Token Factory home</summary>

![Token Factory home](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/token-factory-home.png)

</details>

<details>
<summary>Show screenshot: API Key creation</summary>

![API Key creation](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/api-key-creation.png)

</details>

Now we store that key somewhere our code can find it. We use a `.env` file: a plain text file that holds keys as `NAME=value` lines.
**Create it:**

1. In the same folder as this notebook, create a folder called `lesson` (if it doesn't exist yet).
2. Inside it, create a new plain text file named exactly `.env` (note: no name before the dot!). In VS Code or Jupyter: right-click the `lesson` folder, New File, type `.env`. From a terminal: `touch lesson/.env`.
3. Open it and paste in your key:

```
NEBIUS_API_KEY=paste-your-key-here
```

4. Save. That's it. No quotes, no spaces around the `=`.

**Why a `.env` file at all?** In real development, configuration that changes between people and machines (and *especially* secrets) lives outside your code. The `.env` file holds it; `load_dotenv()` reads it into environment variables. That way the same code runs on your laptop, your teammate's, and the server, and no key ever gets committed. **Key hygiene reminder:** keys live in `.env`, never in code, and `.env` goes in `.gitignore`. Never commit a notebook with a real key in it, unless you want someone stealing your Token Factory credits!

**Running in Google Colab?** Colab doesn't have your `.env`. Two options:

- Upload it: open the file browser (folder icon, left side) and drag your `.env` in, then point `load_dotenv()` at it with load_dotenv("path-to-your-env-file").
- Or use **Colab Secrets** (key icon, left side): add `NEBIUS_API_KEY` there, then run `from google.colab import userdata; os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")` instead of `load_dotenv()`.

One common gotcha: your operating system may hide files starting with a dot. If the file seems to vanish after you create it, it's still there. Your code will find it even if Finder or Explorer doesn't show it!

In [2]:
%pip install -q openai python-dotenv rich sympy

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os

from dotenv import load_dotenv
from rich import print

load_dotenv("lesson/.env")
%load_ext rich

# Colab users: comment the two lines above and use Colab Secrets instead:
# from google.colab import userdata
# os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")

assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in lesson/.env"
print("Keys loaded.")

Keys loaded.

## 3. The model catalog

The model catalog is the home for all models on Token Factory. Here, you can view basic information, and types of availability.

<details>
<summary>Show screenshot: Model catalog</summary>

![Model catalog](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-catalog.png)

</details>

The catalog hosts frontier open models: **GLM 5.2**, **Kimi 3** and the **NVIDIA Nemotron family**, among others.

## 4. The NVIDIA Nemotron ecosystem

Our flagship models for this course are the **NVIDIA Nemotron** series, a family of open-source models from NVIDIA's. The Nemotron series is NVIDIA's latest and most impressive models - spanning both small and efficient to large and frontier-performing.  The family shares training lineage and behavior, so you can prototype on a small one and scale up without rewriting anything. The three we care about:

| | **Nemotron 3 Nano** | **Nemotron 3 Super** | **Nemotron 3 Ultra** |
|---|---|---|---|
| Routing key | `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B` | `nvidia/nemotron-3-super-120b-a12b` | `nvidia/Nemotron-3-Ultra-550b-a55b` |
| Size | Smallest | Medium | Largest |
| Rough character | Fast and very cheap; great for everyday tasks, classification, drafting | The middle ground; strong general model for most workloads | The heavyweight; deepest reasoning, highest price |

A useful mental model: **start small, scale only when the task needs it.** The bigger the model, the more it costs per token and the slower it generates - and for a surprising share of real work, the small one is simply good enough!

**Our workhorse for this chapter is Nemotron 3 Nano.** We'll bring in its bigger siblings when we want a comparison, which will mimmic how you should scale with a real application.

## 5. Reading a model card

If you were deciding on an engine to be put in a car, you would likely read through different spec sheets until you found the one to fit your needs.

Every model in the catalog has a card - this is its spec sheet. The model cards are something you will come back to constantly: for the model string, for prices, for context limits, for what the model can and can't do. Need a fast car? You will pick the strongest - and likely most expensive - option. If you care about efficiency, you would choose one that is smaller and cheaper. The same choices will persist for the models you choose.

Click into **NVIDIA-Nemotron-3-Nano-30B-A3B**, scroll down, and click the model card.

<details>
<summary>Show screenshot: Nemotron 3 Nano model card</summary>

![Nemotron 3 Nano model card](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-card.png)

</details>

### Copying the model string

At the top of the endpoint properties you'll find the **routing key** - for our model it's `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B`, with a copy icon right next to it. **That exact string is what you pass as `model=` in every API call.** Always copy it from the card rather than typing it; a typo here is the single most common first error, and the API's "model not found" message won't tell you which character you got wrong.

### Reading the model card - what specs are important to us?

**Context window 262k.**
Find this number on the Nano card — it's the ceiling on everything the model can see at once: your prompt, the conversation history, any documents you paste in, *and* the output it generates. They all share this budget. Think of it as the model's working memory. A bigger window means you can hand it an entire codebase or a stack of contracts in one go; a smaller one means you'll have to chop things up and manage what the model gets to see.

**Pricing.**
Pricing is in terms of tokens, and there is a reason for the separation. Input tokens are cheap because the model ingests your whole prompt in a single go; output tokens cost more because the model generates them one at a time. Each one is a full trip through the model. Practical consequence: a chatty model that writes long answers costs you much more (think, what can we do to outputs to reduce this?). For scale: Ultra charges \$1.00 / \$3.00 for the same million tokens — roughly **15× more**.

**Modality: Text-to-text.**
What goes in and what comes out. Our model reads text and writes text, but scroll through the catalog and you'll see vision models (they accept images) and embedding models (they output vectors, not sentences). Checking this field takes two seconds and saves you from building half a project around a model that can't do what you assumed.

**Tool calling / Reasoning: Available.**
We'll return to what these mean later in the course — for now, just note that they're on the card, and that they'll matter a lot once we build agents.

**License.**
Open-weight does not mean do-whatever-you-want. Licenses differ on commercial use, redistribution, and what you can train on the outputs. Important to understand before putting a model in production, or starting work on something you may not be allowed to do.

### Exercise: pick a model by reading its card

Open two cards side by side: **`nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B`** and **`nvidia/Nemotron-3-Ultra-550b-a55b`**.

Two jobs. Tor each, decide which model you'd pick, and name the **one or two card fields** that settle it. One sentence of justification each ("it's bigger" doesn't count!):

1. **An everyday assistant** — rephrases emails, drafts short summaries, answers quick questions all day long.
2. **A contract analyst** — answers questions about 300-page legal documents pasted in whole, where a wrong answer is very costly.

*Hint: for one of these, the price gap is important. For the other, look at the context window and ask what a mistake costs.*

*MOsokina's comments:*

**The model choice*

1. **An everyday assistant** — 
**`nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B`**
- tasks are small but high volume, the pricing field decides: Nano is about 15x cheaper per token.
1. **A contract analyst** 
**`nvidia/Nemotron-3-Ultra-550b-a55b`**.
 -  the prompt is big (large document plus questions and answers), so context window field matters as a wrong answer is costly the Ultra's complex reasoning is worth.
   

## 6. First calls in the UI (Playground)

Before any code, use the **Token Factory playground**:

1. Open the model card for **Nemotron 3 Nano** and click **Go to playground**.
2. Send this prompt:

   > *A survey shows 62% of respondents prefer Product A and 38% prefer Product B, but the group that prefers A is three times smaller in the overall population. Which product actually wins overall? Show your reasoning briefly.*

3. Look at the **metrics below the response** - response time, token counts (it is OK to not fully understand these yet!).
<details>
<summary>Show screenshot: Nemotron response metrics</summary>

![Nemotron response metrics](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/request-metrics.png)

</details>

4. Now run the **exact same prompt** on **Nemotron 3 Ultra** side by side.

**What to compare:** the number of **output tokens** each model spent, how long each took, and the answers themselves - does the big model reason more carefully, or just write more? Did the small one get the right answer anyway? This is your first *evaluation*, and the honest answer to "which model is better" always starts with "for what purpose?".

*Mosokina's comments*

**The results:**
- Quality: both answers are correct. Both read "three times smaller" as one-third, and both notice that the phrase is ambiguous.
- Reasoning: Ultra's thinking is about twice as long. It re-checks its understanding several times, but the extra tokens do not change the answer. Nano does not go back and re-read its interpretation.
- Speed:

| | Ultra | Nano |
|---|---|---|
| First token | 463 ms | 234 ms |
| Total time | 3 s | 3.8 s |
| Speed | 354.6 t/s | 224.2 t/s |

Ultra does more reasoning and takes longer to produce its first token, but it generates tokens faster (354.6 vs 224.2 t/s), so it finishes earlier than Nano (3 s vs 3.8 s).

**Conclusion:** for this prompt Nano is good enough — same answer at roughly 15× lower price.


## 7. The Chat Completions request

The Playground isn't for real applications, it's for *you* to evaluate a model. To use a model in real workflows, you need to call it from code, and the Token Factory API is exactly for that.

A Chat Completions request is the standard way to interact with models via API. It is a structured format for how we communicate to models and how they communicate back. A sample completions request for your model can be seen on the model card:

<details>
<summary>Show screenshot: Chat Completions example</summary>

![Chat Completions example](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-card-completions.png)

</details>

### Token Factory's API is OpenAI-compatible

If you ever went through any OpenAI API tutorial, this will be easy-peasy for you: everything works the same way, you just change the `base_url` and the model names. That compatibility is a big deal in practice:

- The entire ecosystem of tools works out of the box
- Drop-in migration from other providers
- Every tutorial on the internet works with a **one-line `base_url` change**

### 7.1 Anatomy of the request

Before the code, the pieces:

- **`model`** *(required)* — the routing key you copied from the card.
- **`messages`** *(required)* — the conversation as a list of roles. **Key note: the API is stateless — you send the whole conversation history every time.** The roles:
  - **`system`** — instructions for how the model should behave. Less about giving it expertise, more about **tone of voice and style**: the same question answered by "a formal analyst" and "a casual intern" reads completely differently. We'll see this live in a minute. Set once, at the top.
  - **`user`** — what the human says. Your prompts go here!
  - **`assistant`** — what the model said. You append its previous replies here so it remembers the conversation.
  - **`tool`** — results from function calls, which we'll meet in the next notebook.
- **Sampling params** *(optional)* — `temperature`, `top_p`, `max_tokens`. We will not worry about these for now.
- **The response object** — what comes back. We'll open one up right after the first call.

### 7.2 Minimal call

The model card provides a ready-made snippet ("Use model in code" - the one below is adapted from it), so you never have to write this boilerplate from memory:

In [6]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.us-central1.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "In two sentences, what does NVIDIA's Nemotron model family target?"

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

print(response.choices[0].message.content)

NVIDIA's Nemotron model family targets the development of high‑performance, generative AI models optimized for 
enterprise‑grade workloads, especially in simulation, digital twin, and large‑scale language‑model fine‑tuning. It 
is engineered to deliver scalable, cost‑effective AI inference and training across diverse industry use cases such 
as healthcare, automotive, and robotics.

### 7.3 What came back? A one-time tour of the response

The response isn't just text - it's a structured object, and a few of its fields matter for everything we do later. Let's dump the whole thing **once**, look around, and then never print it in full again:

In [7]:
# One-time tour — after this cell, we only ever print .content
print(response.to_json())

# The fields worth remembering:
print("---")
print("content:       ", response.choices[0].message.content[:60], "...")   # the answer
print("finish_reason: ", response.choices[0].finish_reason)                  # why it stopped ("stop" = natural end, "length" = hit max_tokens)
print("usage:         ", response.usage)                                     # tokens in/out — this becomes our COST signal in Notebook 3

{
  "id": "chatcmpl-2995c6ef-b0fb-4a3e-8050-81d7cf9d4bee",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "\nNVIDIA's Nemotron model family targets the development of high‑performance, generative AI 
models optimized for enterprise‑grade workloads, especially in simulation, digital twin, and large‑scale 
language‑model fine‑tuning. It is engineered to deliver scalable, cost‑effective AI inference and training across 
diverse industry use cases such as healthcare, automotive, and robotics.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [],
        "reasoning": "We need to answer in two sentences, describing what NVIDIA's Nemotron model family targets. 
Provide concise answer.\n"
      },
      "stop_reason": null,
      "token_ids": null,
      "routed_experts": null
    }
  ],
  "created": 1791892613,
  "model": "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
  "object": "chat.completion",
  "service_tier": null,
  "system_fingerprint": "vllm-0.22.0-b08c96cd",
  "usage": {
    "completion_tokens": 104,
    "prompt_tokens": 44,
    "total_tokens": 148,
    "completion_tokens_details": null,
    "prompt_tokens_details": null
  },
  "moderation": null,
  "prompt_logprobs": null,
  "prompt_token_ids": null,
  "prompt_text": null,
  "kv_transfer_params": null
}

---

content:        
NVIDIA's Nemotron model family targets the development of h ...

finish_reason:  stop

usage:         
CompletionUsage(
    completion_tokens=104,
    prompt_tokens=44,
    total_tokens=148,
    completion_tokens_details=None,
    prompt_tokens_details=None
)

Three things to file away: **`content`** is the answer, **`finish_reason`** tells you *why* generation stopped (watch for `"length"` - it means the answer was cut off), and **`usage`** counts prompt and completion tokens - the number your bill is computed from. From here on, our cells print only the content, for readability.

### 7.4 System prompts: same question, two voices

The claim above was that system prompts are mostly about tone and style. Prove it! Same user question, two different personas:

In [6]:
QUESTION = "Is the specialty coffee market growing?"

for persona in [
    "You are a formal senior market analyst. Precise, measured, no exclamation points.",
    "You are an enthusiastic intern on the market-research team. Casual, punchy, emoji welcome.",
]:
    resp = client.chat.completions.create(
        model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
        messages=[
            {"role": "system", "content": persona},
            {"role": "user", "content": QUESTION},
        ],
    )
    print(f"[bold]--- {persona[:40]}... ---[/bold]")
    print(resp.choices[0].message.content, "\n")

--- You are a formal senior market analyst. ... ---

The global specialty coffee market is experiencing consistent revenue growth, with a compound annual growth rate 
(CAGR) of approximately 5-8% over the past five years, according to industry reports from sources including the 
Specialty Coffee Association (SCA) and Statista. Growth is driven by rising consumer demand for premium, ethically 
sourced products, increased café culture adoption in urban centers, and expanding retail penetration through 
specialty grocers and e-commerce. Key growth regions include North America and Europe, with emerging markets in 
Asia-Pacific showing accelerated expansion. However, this growth occurs within a context of rising input costs and 
inflationary pressures, which are compressing margins for some players. The market is projected to reach $50 
billion+ in global revenue by 2027, reflecting sustained demand for differentiated coffee experiences.

--- You are an enthusiastic intern on the ma... ---

1. CAGR  
2. Demand  
3. Premium  
4. Urban  
5. Growth  
6. Trend  
7. Consumption  
8. Revenue  
9. Expansion  
10. Popularity  

***  
The specialty coffee scene is booming – people are willing to pay more for that perfect brew ☕️✨.  
Demand is spiking in cities and suburbs alike, driven by Millennials and Gen Z love for quality.  
Even the pandemic couldn’t slow it down; it just pushed sales online 📈.  

***  
- CAGR  
- Demand  
- Premium  

***  
The market’s definitely on an upward trajectory, and the numbers back it up 💡.  
Every new coffee shop seems to be chasing that specialty badge, which fuels the growth 🌱.  
So, if you’re wondering whether it’s growing – absolutely, and it’s only getting hotter 🔥.  
Stay tuned, the next wave of coffee innovation is just around the corner 🚀.

Same facts, completely different voice. In an agent, the system prompt is where you pin down *how* it should behave - style, format, boundaries, etc.

### 7.5 Multi-turn: the API is stateless, so *you* carry the memory

Ask a follow-up question in a fresh request and the model has no idea what you're talking about - nothing is stored server-side. "Memory" is literally you resending the conversation. First, fully by hand, so you see every move:

In [7]:
# Turn 1: ask
history = [
    {"role": "system", "content": "You are a concise assistant for a market-research team."},
    {"role": "user", "content": "Name three fast-growing beverage categories."},
]
resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
answer_1 = resp.choices[0].message.content
print(answer_1)

# Turn 2: append the model's answer, then our follow-up, then resend EVERYTHING
history.append({"role": "assistant", "content": answer_1})
history.append({"role": "user", "content": "Which of those is cheapest to enter as a small brand?"})

resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
print("---")
print(resp.choices[0].message.content)   # it knows "those" only because we resent the history

- Hard seltzers (spiked water/ready‑to‑drink low‑calorie cocktails)  
- Ready‑to‑drink (RTD) teas and coffees  
- Functional non‑alcoholic beverages (e.g., adaptogenic drinks, infused waters, sparkling probiotic sodas)

---

Functional non‑alcoholic beverages (e.g., infused waters, sparkling adaptogenic drinks) are generally the cheapest 
to launch as a small brand because they require lower regulatory overhead, minimal alcohol‑related licensing, and 
can be produced with simpler formulations and packaging.

That works, but the appending pattern is always the same, so wrap it in a tiny loop and you have a chat:

In [8]:
history = [{"role": "system", "content": "You are a concise assistant for a market-research team."}]

def ask(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
    answer = resp.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    return answer

print(ask("Name three fast-growing beverage categories."))
print("---")
print(ask("Which of those is cheapest to enter as a small brand?"))

Here are three beverage categories that have shown the strongest growth rates in recent market‑research reports 
(U.S. and global, 2023‑24):

| Category | Why it’s exploding | Typical sub‑segments driving the surge |
|----------|-------------------|----------------------------------------|
| **Functional/Wellness Drinks** | Growing consumer focus on health, immunity, stress‑relief and gut support; 
premiumization of “benefit‑driven” beverages. | Adaptogen teas, sparkling probiotic waters, plant‑based protein 
drinks, low‑sugar kombucha, caffeine‑boosting teas (e.g., matcha, guayusa). |
| **Ready‑to‑Drink (RTD) Coffee & Cold Brew** | Convenience meets coffee culture; younger consumers (Gen Z, 
Millennials) seek café‑quality drinks on the go and are willing to pay a premium. | Cold‑brew bottled coffees, 
nitro coffees, coffee‑based cocktails (e.g., “coffee‑infused seltzers”), caffeine‑plus‑no‑sugar mixes. |
| **Non‑Alcoholic & Low‑ABV “Mocktail” Spirits** | “Sober curious” movement, regulatory pressure on alcohol 
marketing, and the rise of social “drink‑free” gatherings; brands are offering sophisticated flavor profiles that 
mimic spirits. | Alcohol‑free gin, whiskey‑style “spirit‑alternatives,” low‑ABV canned cocktails, functional 
mocktails with adaptogens or botanics. |

**Quick takeaway:** These categories are expanding at double‑digit CAGR (often 12‑20 % YoY) and are attracting both
established CPG players and niche startups, making them attractive targets for product innovation, distribution 
expansion, and partnership opportunities.

---

**The “cheapest” entry point among the three is **`Functional/Wellness Drinks`**.**  
Below is a quick cost‑breakdown and why they tend to be the most budget‑friendly for a scrappy brand.

| Cost Driver | Functional/Wellness Drinks | RTD Coffee/Cold Brew | Non‑Alcoholic “Mocktail” Spirits |
|------------|---------------------------|----------------------|--------------------------------|
| **Ingredient sourcing** | Bulk herbs, fruits, probiotics, adaptogens can be bought from wholesale suppliers at < 
$0.30 / L. | Pre‑brew coffee concentrate and nitro equipment are pricey; beans cost $0.50‑$0.80 / L and require 
fresh‑brew passes. | Botanical extracts and neutral spirits‑alternatives often need costly aroma‑chemistry 
(essential oils, flavor‑carriers). |
| **Regulatory hurdles** | Generally food‑grade; only need GRAS status for novel ingredients, plus standard label 
claims (no FDA pre‑approval). | Same as above, but if you add caffeine‑boosts or functional additives, you must 
file a **New Dietary Ingredient (NDI)** notification. | Must meet **alcohol‑free** (<0.5 % ABV) rules *and* 
beverage‑alcohol labeling; many markets treat them as “spirit‑alternatives” and require a separate “spirit” 
registration, adding paperwork. |
| **Packaging simplicity** | 12‑oz PET or 1‑L glass bottles + simple sleeve; can use co‑packers that fill and cap 
in one step. | Often sold in 12‑oz cans or 1‑L glass that need **nitro** (CO₂‑infused) lines for premium look → 
higher co‑packer fees. | Typically sold in 8‑oz cans or small glass “mini‑sprits”; cans are cheap, but you must 
design a label that complies with *alcohol‑free* regulations and may need a **“spirit‑style”** statement. |
| **Minimum order volume** | Many co‑packers accept **as low as 5 k–10 k units** for a new SKU (especially with 
private‑label fruit bases). | Even the cheapest co‑packer runs start at ~25 k–50 k cans because of equipment setup.
| Similar to coffee: need at least 15 k‑20 k cans for first run; plus you need a separate label compliance file for
“spirit‑like” branding. |
| **Marketing entry cost** | Positioning is straightforward: “functional + flavor” → can leverage existing wellness
influencers without a complex story. | Requires positioning around “café‑quality on‑the‑go” → competition is 
fierce, requiring bigger ad spend to cut through. | Requires educating consumers about “sober‑curious” lifestyle; 
often needs a narrative campaign to justify a “spirit‑like” product. |
| **Overall first‑run cost estimate** (per 12‑oz unit) | **$0.45 – $0.80** (ingredients + co‑pack + label) | 
**$0.70 – $1.20** (incl. higher packaging & equipment fees) | **$0.85 – $1.35** (includes alcohol‑free 
classification compliance) |

### Why Functional/Wellness Wins the “Cheapest” Crown

1. **Low‑cost raw materials** – You can start with simple, high‑volume ingredients (e.g., ginger‑lemon, 
hibiscus‑berry, plain sparkling water) that are cheap when bought in bulk.
2. **No alcohol‑level regulations** – Avoiding the 0.5 % ABV threshold eliminates a whole compliance pathway and 
the need for a separate “spirit” license in many jurisdictions.
3. **Flexible labeling** – Claims such as “supports gut health” or “adaptogenic boost” are allowed with minimal 
scientific substantiation, reducing R&D spend.
4. **Co‑packer friendliness** – Many beverage copackers already have fruit‑infusion lines; they can add functional 
powders or extracts on‑the‑fly, keeping setup costs low.
5. **Small‑batch viability** – You can launch with 5‑10 k units, test three flavors, and iterate without massive 
sunk costs.

### Practical Steps to Get Started on a Shoestring Budget

| Step | What to Do | Approx. Cost |
|------|------------|--------------|
| **1️⃣ Define a niche sub‑segment** | e.g., “Probiotic Sparkling Kombucha,” “Adaptogen‑Infused Herbal Seltzer,” or
“Low‑Sugar Functional Iced Tea.” | $0 |
| **2️⃣ Source a white‑label base** | Look for co‑packers offering “private‑label sparkling water + functional 
powde

Notice what the loop implies about cost: every turn resends the *whole* history, so long conversations re-pay for everything already said.

### Exercise: try to create a request that does these tasks!

All of these use our market-research assistant. Fill in the `## What goes here?` gaps.

#### 1. Only produce 10 tokens or less per response

In [13]:
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Give me a one-line headline on the energy-drink market."

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
    max_tokens=10,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print(response.choices[0].message.content)
print("---")
print("finish_reason:", response.choices[0].finish_reason)  # what do you expect to see here?

Energy‑drink market soars 12

---

finish_reason: length

#### 2. Change the *voice*: make the assistant answer like a skeptical veteran analyst who distrusts hype

In [8]:
SYSTEM_PROMPT = "you are a skeptical veteran analyst who distrusts hype."
USER_MESSAGE = "Is the specialty coffee market growing?"

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

print(response.choices[0].message.content)

**Short answer:** Yes—the specialty‑coffee segment has been expanding at a double‑digit pace for the last several 
years—but the numbers are more nuanced than most headlines suggest.

---

### The data points I keep on my desk

| Metric (global) | Latest figure (2023‑24) | 5‑year CAGR* |
|-----------------|------------------------|--------------|
| **Definition of “specialty coffee”** (SCA‑graded 80 pts + or equivalent) | ≈ 30 % of total coffee‑bean volume 
sold in developed markets; ≈ 12 % of total global bean volume | ~ 9 % (2019‑2024) |
| **Retail sales of specialty‑grade beans** | US $ 45 bn (U.S. alone) | ~ 12 % CAGR (2019‑2024) |
| **Consumer‑survey intent to purchase** (specialty cafés, beans, equipment) | 38 % of U.S. adults say they’d pay ≥
$ 5 for a specialty drink | + 5 pp YoY |
| **Import share of “premium” beans** (import‑weighted by HS 0901‑0904) | 13 % of total coffee imports (by value) |
+ 4 % YoY |
| **Chain‑level penetration** (e.g., Starbucks “Reserve”, Costa “Premium” lines) | ~ 15 % of total chain sales; 
growth slowed to ~ 2 % YoY in 2023 | 2 % YoY (2023‑24) |

\*CAGR calculated on reported sales, not on “units” – a critical distinction when price premiums are rising faster 
than volume.

---

### Why I’m skeptical of the “boom” narrative

1. **Premium pricing masks real volume growth**  
   The headline CAGR numbers above are driven as much (if not more) by the 30‑50 % price premium over commodity 
beans as by actual unit growth. When the market pushes higher‑margin products, reported dollar‑growth rates can 
look spectacular even if the underlying kilogram count is flat or slightly declining.

2. **Geography matters—most of the growth is concentrated in a handful of markets**  
   - **U.S., Canada, Western Europe, Japan, South Korea, and a handful of affluent Asian metros** account for > 70 
% of specialty‑coffee revenue.  
   - In many developing regions, “specialty” is still a niche; price elasticity is low, and consumers gravitate 
toward value rather than price‑premium coffee.

3. **Retail channel saturation is creeping in**  
   - The explosion of boutique cafés in the early 2010s has largely run its course. New openings now tend to be 
strategic “fill‑in” locations rather than aggressive roll‑outs.  
   - Same‑store sales growth for chain‑owned specialty concepts is now largely organic, hovering around 1‑2 % YoY.

4. **Supply‑side constraints are under‑reported**  
   - Specialty‑grade beans are more sensitive to climate shocks (e.g., droughts in Brazil’s high‑altitude regions, 
pest outbreaks in Central America).  
   - The “single‑origin” and “direct‑trade” models rely on relatively small farms; scaling them without diluting 
quality is a logistical nightmare.

5. **Consumer fatigue and “premium fatigue”**  
   - After a decade of “third‑wave” branding, many coffee drinkers feel they’ve been sold the same narrative 
repeatedly.  
   - The rise of “ready‑to‑drink” (RTD) canned coffee, oat‑milk lattes, and home‑brew kits is cannibalising 
traditional specialty‑café traffic in some urban cores.

---

### What would make the growth **more legitimate** in my eyes?

| Condition | Why it matters |
|-----------|----------------|
| **Sustained volume growth ≥ 5 % CAGR** across **multiple regions** (not just price‑driven dollars) | Shows the 
market is expanding, not just re‑packaging the same beans. |
| **Transparent supply‑chain metrics** (e.g., traceability to *certified* farms, measurable farmer‑income uplift) |
Turns “specialty” from a marketing label into a verifiable impact story. |
| **Premium‑price erosion** (i.e., consumers buying specialty at *smaller* price premiums over time) | Indicates 
that the value proposition is moving from “novelty” to “necessity.” |
| **Diversification of product formats** (e.g., ready‑to‑drink specialty, at‑home brewing kits, low‑acid blends) | 
Reflects adaptation to changing consumption habits rather than reliance on café footfall. |

---

### Bottom line (in my ve

#### 3. Challenge! Make the model produce 10 identical responses with:

```python
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Produce 5 bullet points on why cold brew is winning the coffee market."
```

**Hint: you may need to use a loop and a certain parameter that makes the model more deterministic (less creative)**

*MOsokina's note*

 temperature=0  makes model always pick the most likely token; 
 seed=42 acks as backup: it makes the model repeat the same choice in case of any randomness.

In [19]:
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Produce 5 bullet points on why cold brew is winning the coffee market."

responses = []

for n in range(10):
    resp = client.chat.completions.create(
        model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_MESSAGE},
        ],
        temperature=0,
        seed=42
    )
    responses.append(resp.choices[0].message.content)

print(responses[0])
print("distinct responses:", len(set(responses)))

print(f"[bold]--- RESPONSE No {n+1}... ---[/bold]")
print(resp.choices[0].message.content, "\n")

- **Smooth, low‑acid profile** appeals to health‑conscious and sensitive‑stomach consumers, driving repeat 
purchases.  
- **Convenient ready‑to‑drink formats** (bottles, cans) fit the on‑the‑go lifestyle and dominate grocery‑store 
shelf space.  
- **Premium perception & “craft” positioning** allows higher price points and attracts millennials/Gen Z willing to
pay for specialty experiences.  
- **Extended shelf life & easy storage** reduce waste for retailers and simplify distribution compared with 
hot‑brew coffee.  
- **Versatile flavor innovations** (nitro, flavored infusions, plant‑based milks) keep the category fresh and 
encourage trial across diverse taste preferences.

distinct responses: 10

--- RESPONSE No 10... ---

- **Smooth, low‑acid profile** appeals to health‑conscious and sensitive‑stomach consumers.  
- **Convenient ready‑to‑drink formats** (bottles, cans) fit on‑the‑go lifestyles and boost shelf‑life sales.  
- **Premium perception & “craft” positioning** allows higher price points and brand differentiation.  
- **Versatile serving options** (milk, sweeteners, flavor infusions) attract a broad, customizable audience.  
- **Strong social‑media buzz & influencer culture** drives trial and sustained popularity among younger 
demographics.

## Wrap-up!

You created an account and an API key, and stored it in a dot-env file so it stays out of your code. 

You learned to read a model card - the routing key, the context window and the three things sharing it, the two pricing rates and why output costs more, modality, and the license. 

You tested a model in the playground and compared a small one against a large one on the same prompt. 

And then you built the same interaction in code: a single request, the response object with its content, finish reason, and usage, a system prompt to control voice, and a multi-turn conversation where you carry the history yourself.

Together, that means you can choose a model for a job, call it from Python, and hold a real conversation with it while knowing exactly what each request cost you.

You now have a working foundation - a model you've chosen deliberately, and the code to talk to it.
